[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gonzalezulises/formacion-docente-bi-faces/blob/main/modulo-03-business-intelligence/notebooks/03_02_power_bi_fundamentos.ipynb)

# 03.02 — Introducción a Power BI

**Módulo 3: Business Intelligence aplicado a decisiones**  
**Instructora:** Ilda Rojas  
**Duración:** 2 horas

---

> **Nota importante:** Este notebook es una guía teórica y de ejercicios guiados. Los participantes deben seguir los pasos en **Power BI Desktop** (Windows) simultáneamente. Si no dispone de Windows, puede usar **Tableau Public** o **Google Looker Studio** como alternativa para los conceptos equivalentes.

## Objetivos de aprendizaje

Al finalizar esta sesión, el participante será capaz de:

1. Importar y transformar datos usando Power Query
2. Comprender el modelo de datos (tablas de hechos y dimensiones)
3. Crear medidas básicas con DAX
4. Diseñar un dashboard interactivo con las visualizaciones principales
5. Aplicar filtros e interacciones entre visualizaciones

---

## 1. ¿Qué es Power BI?

Power BI es una plataforma de Business Intelligence de Microsoft que permite:

- **Conectar** a múltiples fuentes de datos (Excel, CSV, bases de datos, APIs)
- **Transformar** datos sin programar (Power Query)
- **Modelar** relaciones entre tablas
- **Visualizar** con gráficos interactivos
- **Compartir** dashboards con la organización

### Componentes principales

| Componente | Función | Analogía |
|---|---|---|
| **Power Query** | ETL (Extraer, Transformar, Cargar) | La "cocina" donde se preparan los datos |
| **Modelo de datos** | Relaciones entre tablas | El "esqueleto" que conecta todo |
| **DAX** | Cálculos y medidas | Las "fórmulas" (similar a Excel pero más potente) |
| **Visualizaciones** | Gráficos e interacciones | La "presentación" que ve el usuario |

### Versiones

| Versión | Costo | Uso |
|---|---|---|
| Power BI Desktop | Gratuito | Crear reportes (solo Windows) |
| Power BI Service | Licencia Pro/Premium | Compartir y colaborar en la nube |
| Power BI Mobile | Gratuito | Consumir reportes en móvil |

---

## 2. Power Query: conexión y transformación de datos

Power Query es el editor de transformación de datos de Power BI. Funciona con una interfaz visual (sin código) aunque por detrás genera código en lenguaje M.

### 2.1 Conectar a un archivo CSV

**Pasos en Power BI Desktop:**

1. Abrir Power BI Desktop
2. Clic en **Inicio → Obtener datos → Texto/CSV**
3. Seleccionar el archivo `rendimiento_academico.csv`
4. Verificar la vista previa: ¿se detectaron correctamente las columnas?
5. Clic en **Transformar datos** (no en "Cargar" directamente)

### 2.2 Transformaciones básicas comunes

Una vez en el editor de Power Query:

| Transformación | Cómo hacerlo | Por qué |
|---|---|---|
| **Cambiar tipo de dato** | Clic en ícono de columna → seleccionar tipo | Asegurar que números sean números, fechas sean fechas |
| **Renombrar columnas** | Doble clic en encabezado | Nombres claros para las visualizaciones |
| **Eliminar columnas** | Clic derecho → Quitar | Reducir datos a lo necesario |
| **Filtrar filas** | Clic en flecha del encabezado → filtrar | Excluir datos irrelevantes o erróneos |
| **Reemplazar valores** | Clic derecho en columna → Reemplazar valores | Corregir inconsistencias ("Admin" vs "Administración") |
| **Dividir columna** | Clic derecho → Dividir columna | Separar "2023-I" en año y período |

### 2.3 Ejercicio guiado: Preparar rendimiento_academico.csv

Siga estos pasos en Power BI:

1. Importar `rendimiento_academico.csv`
2. En Power Query:
   - Verificar que las columnas numéricas sean tipo "Número decimal"
   - Verificar que las columnas de texto sean tipo "Texto"
   - Renombrar columnas si tienen nombres técnicos poco claros
   - Revisar si hay valores nulos y decidir cómo manejarlos
3. Clic en **Cerrar y aplicar**

In [ ]:
# Veamos la estructura del dataset que usaremos en Power BI
import pandas as pd

df = pd.read_csv('../datasets/universidad/rendimiento_academico.csv')
print("Estructura del dataset:")
print(f"  Filas: {df.shape[0]}")
print(f"  Columnas: {df.shape[1]}")
print()
print("Tipos de datos:")
print(df.dtypes)
print()
print("Primeras 5 filas:")
df.head()

In [ ]:
# Valores nulos por columna
print("Valores nulos:")
print(df.isnull().sum())
print()
print("Valores únicos por columna categórica:")
for col in df.select_dtypes(include='object').columns:
    print(f"  {col}: {df[col].nunique()} valores únicos → {list(df[col].unique()[:5])}")

---

## 3. Modelo de datos: tablas de hechos y dimensiones

En Business Intelligence, los datos se organizan en un **modelo dimensional** que facilita el análisis.

### 3.1 Conceptos clave

| Concepto | Definición | Ejemplo FACES |
|---|---|---|
| **Tabla de hechos** | Contiene las mediciones/eventos | Inscripciones, calificaciones, pagos |
| **Tabla de dimensiones** | Contiene atributos descriptivos | Estudiantes, carreras, períodos, docentes |
| **Clave primaria** | Identifica únicamente cada fila en una dimensión | ID_estudiante, código_carrera |
| **Clave foránea** | En la tabla de hechos, referencia a la dimensión | carrera_id en la tabla de calificaciones |

### 3.2 Esquema estrella simplificado

Para nuestro dataset de rendimiento académico, el esquema sería:

```
                    ┌──────────────┐
                    │  Dim_Carrera  │
                    │──────────────│
                    │ codigo       │
                    │ nombre       │
                    │ facultad     │
                    └──────┬───────┘
                           │
┌──────────────┐    ┌──────┴───────┐    ┌──────────────┐
│ Dim_Período  │    │   HECHOS:    │    │Dim_Estudiante│
│──────────────│────│ Rendimiento  │────│──────────────│
│ periodo      │    │──────────────│    │ id           │
│ año          │    │ nota         │    │ nombre       │
│ semestre     │    │ asistencia   │    │ sexo         │
└──────────────┘    │ estado       │    │ edad         │
                    └──────────────┘    └──────────────┘
```

### 3.3 En Power BI

Con un solo CSV, Power BI crea una sola tabla. Para crear un modelo dimensional:

1. En Power Query, duplicar la tabla
2. En cada copia, quedarse solo con las columnas de esa dimensión
3. Eliminar duplicados para crear la dimensión
4. En la vista de Modelo, crear relaciones arrastrando campos

> **Para este curso:** Trabajaremos con una tabla plana (todo en una tabla) por simplicidad. El esquema estrella es la práctica recomendada para proyectos reales.

---

## 4. DAX básico: fórmulas para análisis

DAX (Data Analysis Expressions) es el lenguaje de fórmulas de Power BI. Es similar a las fórmulas de Excel pero opera sobre tablas completas.

### 4.1 Medidas vs Columnas calculadas

| Tipo | Cuándo usar | Se calcula | Ejemplo |
|---|---|---|---|
| **Medida** | Agregaciones que cambian con filtros | Al momento de visualizar | Promedio de notas (cambia si filtro por carrera) |
| **Columna calculada** | Valor fijo por fila | Al cargar datos | Clasificación aprobado/reprobado |

### 4.2 Funciones DAX esenciales

#### Agregaciones básicas

```dax
-- Promedio de notas
Promedio Notas = AVERAGE(Rendimiento[nota])

-- Total de estudiantes
Total Estudiantes = COUNT(Rendimiento[id_estudiante])

-- Estudiantes únicos
Estudiantes Únicos = DISTINCTCOUNT(Rendimiento[id_estudiante])

-- Suma de créditos
Total Créditos = SUM(Rendimiento[creditos])
```

#### CALCULATE: la función más importante

CALCULATE modifica el contexto de filtro. Permite calcular una medida con filtros adicionales.

```dax
-- Tasa de aprobación
Tasa Aprobación = 
    DIVIDE(
        CALCULATE(COUNT(Rendimiento[id]), Rendimiento[estado] = "Aprobado"),
        COUNT(Rendimiento[id])
    )

-- Promedio solo de aprobados
Promedio Aprobados = 
    CALCULATE(
        AVERAGE(Rendimiento[nota]),
        Rendimiento[estado] = "Aprobado"
    )
```

#### ALL: ignorar filtros

```dax
-- Porcentaje del total (ignora el filtro de carrera)
% del Total = 
    DIVIDE(
        COUNT(Rendimiento[id]),
        CALCULATE(COUNT(Rendimiento[id]), ALL(Rendimiento[carrera]))
    )
```

### 4.3 Buenas prácticas DAX

- Usar `DIVIDE()` en vez de `/` para evitar errores de división por cero
- Nombrar medidas con nombres descriptivos en español
- Agrupar medidas en una tabla de medidas (crear tabla vacía para organizarlas)
- Formatear las medidas: porcentajes como %, números con separador de miles

---

## 5. Visualizaciones principales

### 5.1 Tipos de visualización en Power BI

| Visualización | Uso | Campos necesarios |
|---|---|---|
| **Tarjeta** | KPI individual (un número) | 1 medida |
| **Tarjeta de varias filas** | Varios KPIs juntos | Varias medidas |
| **Gráfico de barras** | Comparar categorías | Eje: categórica, Valor: numérica |
| **Gráfico de líneas** | Tendencias temporales | Eje: fecha/período, Valor: numérica |
| **Tabla / Matriz** | Datos detallados | Múltiples campos |
| **Segmentador** | Filtro interactivo | 1 campo categórico o fecha |
| **Mapa** | Datos geográficos | Ubicación + valor |
| **Treemap** | Composición jerárquica | Grupo + valor |

### 5.2 Filtros e interacciones

Power BI permite tres tipos de filtrado:

1. **Filtros de visualización**: afectan solo un gráfico
2. **Filtros de página**: afectan todos los gráficos de la página
3. **Filtros de informe**: afectan todas las páginas

**Interacciones entre visualizaciones:**

Al hacer clic en una barra de un gráfico, todos los demás gráficos de la página se filtran automáticamente. Esto se llama **cross-filtering** y es una de las características más poderosas de Power BI.

Para configurar interacciones:
- Seleccionar una visualización
- Ir a **Formato → Editar interacciones**
- En cada otra visualización, elegir: Filtrar, Resaltar o Ninguno

---

## 6. Ejercicio paso a paso: Dashboard de matrícula FACES

Vamos a construir un dashboard completo. Siga estos pasos en Power BI Desktop:

### Paso 1: Importar datos

1. Abrir Power BI Desktop
2. Obtener datos → CSV → seleccionar `rendimiento_academico.csv`
3. Transformar datos → verificar tipos → Cerrar y aplicar

### Paso 2: Crear medidas DAX

En la vista de Informe, clic derecho en la tabla → Nueva medida:

```dax
Total Estudiantes = DISTINCTCOUNT(rendimiento_academico[id_estudiante])
```

```dax
Promedio Notas = AVERAGE(rendimiento_academico[nota])
```

```dax
Tasa Aprobación = 
    DIVIDE(
        CALCULATE(
            DISTINCTCOUNT(rendimiento_academico[id_estudiante]),
            rendimiento_academico[estado] = "Aprobado"
        ),
        DISTINCTCOUNT(rendimiento_academico[id_estudiante])
    )
```

### Paso 3: Diseñar el layout

Organizar la página en secciones:

```
┌─────────────────────────────────────────────────────────┐
│  DASHBOARD DE RENDIMIENTO ACADÉMICO — FACES             │
├────────────┬────────────┬────────────┬─────────────────┤
│  Tarjeta:  │  Tarjeta:  │  Tarjeta:  │  Segmentador:  │
│  Total     │  Promedio  │  Tasa      │  Carrera       │
│  Estudiant │  Notas     │  Aprobac.  │  [dropdown]    │
├────────────┴────────────┴────────────┼─────────────────┤
│                                      │  Segmentador:  │
│  Barras: Estudiantes por carrera     │  Período       │
│                                      │  [lista]       │
├──────────────────────────────────────┼─────────────────┤
│                                      │                 │
│  Líneas: Promedio por período        │  Tabla: Top 10 │
│                                      │  mejores notas │
└──────────────────────────────────────┴─────────────────┘
```

### Paso 4: Crear visualizaciones

**Tarjetas KPI (fila superior):**
1. Insertar → Tarjeta → arrastrar "Total Estudiantes"
2. Insertar → Tarjeta → arrastrar "Promedio Notas" → formato: 1 decimal
3. Insertar → Tarjeta → arrastrar "Tasa Aprobación" → formato: porcentaje

**Gráfico de barras:**
1. Insertar → Gráfico de barras agrupadas
2. Eje Y: carrera
3. Eje X: Total Estudiantes (medida)
4. Ordenar descendente

**Gráfico de líneas:**
1. Insertar → Gráfico de líneas
2. Eje X: período
3. Valores: Promedio Notas

**Segmentadores:**
1. Insertar → Segmentador → arrastrar "carrera" → estilo: dropdown
2. Insertar → Segmentador → arrastrar "período" → estilo: lista

### Paso 5: Formato y estilo

- Título de página: "Dashboard de Rendimiento Académico — FACES"
- Colores: usar una paleta consistente (ej: azul institucional)
- Eliminar títulos genéricos de gráficos, reemplazar con títulos descriptivos
- Quitar bordes innecesarios
- Verificar que las interacciones entre gráficos funcionen correctamente

In [ ]:
# Vista previa: así se verían los KPIs con Python
# (En Power BI, estos serían tarjetas interactivas)

import pandas as pd

df = pd.read_csv('../datasets/universidad/rendimiento_academico.csv')

print("="*50)
print("DASHBOARD PREVIEW (datos que verá en Power BI)")
print("="*50)

if 'id_estudiante' in df.columns:
    print(f"\nTotal Estudiantes: {df['id_estudiante'].nunique()}")

if 'nota' in df.columns:
    print(f"Promedio Notas: {df['nota'].mean():.1f}")

if 'estado' in df.columns:
    aprobados = (df['estado'] == 'Aprobado').sum()
    total = len(df)
    print(f"Tasa Aprobación: {aprobados/total*100:.1f}%")

if 'carrera' in df.columns:
    print(f"\nEstudiantes por carrera:")
    print(df['carrera'].value_counts().to_string())

---

## Resumen de la sesión

| Componente | Lo que aprendimos |
|---|---|
| **Power Query** | Conectar CSV, transformar tipos, limpiar datos |
| **Modelo de datos** | Tablas de hechos vs dimensiones, esquema estrella |
| **DAX** | SUM, AVERAGE, COUNT, DISTINCTCOUNT, CALCULATE, DIVIDE, ALL |
| **Visualizaciones** | Tarjetas, barras, líneas, segmentadores, tablas |
| **Interacciones** | Cross-filtering, filtros de página/informe |

### Para practicar

- Importar `matricula_faces.csv` y crear un segundo dashboard
- Experimentar con otros tipos de gráficos: treemap, mapa, matriz
- Crear medidas DAX adicionales: mediana, máximo, mínimo

---

**Próxima sesión:** 03.03 — KPIs para FACES